# Model 2 — Similar Car Finder & Price Range Recommendation (kNN)

**Project Question:** Given my car's specifications, in which price range should I sell my car according to the top 10 cars in the dataset that are most similar to mine?

## Objective
This notebook uses **k-Nearest Neighbors (kNN)** with `NearestNeighbors` to find the most similar cars in the dataset and recommend a fair selling price range based on the prices of those similar cars.

## Method Summary
1. Load the scaled dataset for distance calculation.
2. Load the unscaled dataset to display real car values and prices.
3. Select meaningful similarity features.
4. Fit a kNN model using Euclidean distance.
5. Find the 10 most similar cars for a selected query car.
6. Calculate the recommended price range using **Median ± IQR**.
7. Test multiple `k` values and multiple query cars to evaluate model behavior.

## 1. Import Libraries and Load Data

The scaled dataset is used for kNN distance computation because kNN is distance-based. The unscaled dataset is used to display actual values such as price, year, and mileage.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neighbors import NearestNeighbors

scaled_url = 'https://raw.githubusercontent.com/MamoMGD1/ISE302-DataMining-GroupProject/main/data/processed_dataset.csv'
unscaled_url = 'https://raw.githubusercontent.com/MamoMGD1/ISE302-DataMining-GroupProject/main/data/proceed_dataset_without_scaling.csv'

df_scaled = pd.read_csv(scaled_url)
df_unscaled = pd.read_csv(unscaled_url)

if 'Fiyat' in df_unscaled.columns:
    df_scaled['Fiyat'] = df_unscaled['Fiyat']
elif 'Fiyat_original' in df_scaled.columns:
    df_scaled['Fiyat'] = df_scaled['Fiyat_original']
else:
    raise ValueError("Price column was not found. Please check the dataset columns.")

print(f"Scaled dataset shape: {df_scaled.shape}")
print(f"Unscaled dataset shape: {df_unscaled.shape}")
df_scaled.head()

## 2. Feature Selection

The selected features define what “similar” means. I used year, mileage, engine characteristics, transmission type, body type, fuel type, and damage-related variables.

I also added optional technical features such as weight and maximum speed if they exist in the dataset.

In [ ]:
recommended_features = [
    'Yıl', 'Kilometre', 'Motor Hacmi', 'Motor Gücü', 'Vites Tipi',
    'Kasa Tipi_SUV', 'Kasa Tipi_Crossover',
    'Yakıt Tipi_Dizel', 'Yakıt Tipi_Hibrit',
    'paint_damage_score', 'total_changed_parts',
    'Ağırlık', 'Boş Ağırlığı', 'Maksimum Hız'
]

features = [f for f in recommended_features if f in df_scaled.columns]
X = df_scaled[features].fillna(0)

print(f"Feature matrix shape: {X.shape}")
print("Features used:")
for f in features:
    print("-", f)

## 3. Helper Function — Find Similar Cars and Recommend Price Range

This function fits a `NearestNeighbors` model, finds the nearest cars, displays them, calculates the fair price range, and returns the results.

The recommended price range is calculated as:

**Lower Bound = Median Price − IQR**  
**Upper Bound = Median Price + IQR**

where **IQR = Q3 − Q1**.

In [ ]:
def find_similar_cars(query_idx=42, k=10, show_table=True):
    # Finds k most similar cars for a selected query car and recommends a fair price range.
    if query_idx < 0 or query_idx >= len(df_scaled):
        raise ValueError(f"query_idx must be between 0 and {len(df_scaled)-1}")
    
    knn = NearestNeighbors(n_neighbors=k + 1, metric='euclidean')
    knn.fit(X)
    
    query_car = X.iloc[[query_idx]]
    distances, indices = knn.kneighbors(query_car)
    
    neighbor_indices = indices[0][1:]
    neighbor_distances = distances[0][1:]
    
    neighbor_prices = df_scaled.iloc[neighbor_indices]['Fiyat'].values
    query_price = df_scaled.iloc[query_idx]['Fiyat']
    
    median_price = np.median(neighbor_prices)
    q1 = np.percentile(neighbor_prices, 25)
    q3 = np.percentile(neighbor_prices, 75)
    iqr = q3 - q1
    lower_bound = median_price - iqr
    upper_bound = median_price + iqr
    
    display_cols = [
        'Yıl', 'Kilometre', 'Motor Hacmi', 'Motor Gücü', 'Vites Tipi',
        'Kasa Tipi_SUV', 'Kasa Tipi_Crossover', 'Yakıt Tipi_Dizel', 'Yakıt Tipi_Hibrit',
        'paint_damage_score', 'total_changed_parts', 'Fiyat'
    ]
    display_cols_available = [c for c in display_cols if c in df_unscaled.columns]
    
    neighbors_df = df_unscaled.iloc[neighbor_indices][display_cols_available].copy()
    if 'Fiyat' not in neighbors_df.columns:
        neighbors_df['Fiyat'] = neighbor_prices
    neighbors_df['kNN Distance'] = neighbor_distances
    neighbors_df.index = range(1, k + 1)
    neighbors_df.index.name = 'Rank'
    
    print("=" * 70)
    print(f"Query car index: {query_idx}")
    print(f"k value: {k}")
    print(f"Query car actual price: {query_price:,.0f} TL")
    print("-" * 70)
    print(f"Recommended fair price range: {lower_bound:,.0f} TL — {upper_bound:,.0f} TL")
    print(f"Median of similar cars: {median_price:,.0f} TL")
    print(f"Q1: {q1:,.0f} TL | Q3: {q3:,.0f} TL | IQR: {iqr:,.0f} TL")
    print("=" * 70)
    
    if query_price < lower_bound:
        interpretation = "The query car appears cheaper than similar cars."
    elif query_price > upper_bound:
        interpretation = "The query car appears more expensive than similar cars."
    else:
        interpretation = "The query car price is inside the recommended fair price range."
    print("Interpretation:", interpretation)
    
    if show_table:
        display(neighbors_df.style.format({
            'Fiyat': '{:,.0f} TL',
            'kNN Distance': '{:.4f}'
        }).background_gradient(subset=['Fiyat'], cmap='Blues'))
    
    return {
        'query_idx': query_idx, 'k': k, 'query_price': query_price,
        'neighbor_indices': neighbor_indices, 'neighbor_distances': neighbor_distances,
        'neighbor_prices': neighbor_prices, 'median_price': median_price,
        'q1': q1, 'q3': q3, 'iqr': iqr,
        'lower_bound': lower_bound, 'upper_bound': upper_bound,
        'neighbors_df': neighbors_df, 'interpretation': interpretation
    }

## 4. Main Demo — Query Car with k = 10

For the main demonstration, I selected row index `42` as the query car.

In [ ]:
main_result = find_similar_cars(query_idx=42, k=10, show_table=True)

## 5. Visualization — Price Distribution of Similar Cars

This boxplot shows the price distribution of the 10 most similar cars. The red dot represents the query car's actual price.

In [ ]:
def plot_price_distribution(result):
    neighbor_prices = result['neighbor_prices']
    query_price = result['query_price']
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.boxplot(neighbor_prices, vert=True, patch_artist=True)
    ax.scatter([1], [query_price], zorder=5, s=120, label=f"Query Car: {query_price:,.0f} TL")
    ax.set_ylabel('Price (TL)', fontsize=12)
    ax.set_title(f"Price Distribution of {result['k']} Most Similar Cars", fontsize=14)
    ax.set_xticks([])
    ax.legend()
    plt.tight_layout()
    plt.show()

plot_price_distribution(main_result)

## 6. Visualization — Similarity Distance Bar Chart

Lower distance means the neighbor is more similar to the query car.

In [ ]:
def plot_similarity_distances(result):
    distances = result['neighbor_distances']
    ranks = [f"Neighbor {i+1}" for i in range(len(distances))]
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.bar(ranks, distances)
    ax.set_xlabel('Neighbor Rank', fontsize=12)
    ax.set_ylabel('kNN Distance Score', fontsize=12)
    ax.set_title('Similarity Distances to Nearest Neighbors
(Lower = More Similar)', fontsize=14)
    ax.tick_params(axis='x', rotation=45)
    plt.tight_layout()
    plt.show()

plot_similarity_distances(main_result)

## 7. Experiment 1 — Compare Different k Values

To improve the analysis, I tested different k values: `k=5`, `k=10`, and `k=15`.

In [ ]:
k_values = [5, 10, 15]
k_results = []

for k in k_values:
    result = find_similar_cars(query_idx=42, k=k, show_table=False)
    k_results.append({
        'k': k,
        'Query Price': result['query_price'],
        'Lower Bound': result['lower_bound'],
        'Upper Bound': result['upper_bound'],
        'Median Price': result['median_price'],
        'IQR': result['iqr'],
        'Interpretation': result['interpretation']
    })

k_comparison_df = pd.DataFrame(k_results)
k_comparison_df.style.format({
    'Query Price': '{:,.0f} TL',
    'Lower Bound': '{:,.0f} TL',
    'Upper Bound': '{:,.0f} TL',
    'Median Price': '{:,.0f} TL',
    'IQR': '{:,.0f} TL'
})

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(k_comparison_df['k'], k_comparison_df['Lower Bound'], marker='o', label='Lower Bound')
ax.plot(k_comparison_df['k'], k_comparison_df['Upper Bound'], marker='o', label='Upper Bound')
ax.plot(k_comparison_df['k'], k_comparison_df['Median Price'], marker='o', label='Median Price')
ax.set_xlabel('k Value')
ax.set_ylabel('Price (TL)')
ax.set_title('Effect of k on Recommended Price Range')
ax.legend()
plt.tight_layout()
plt.show()

## 8. Experiment 2 — Test Multiple Query Cars

A stronger project should not depend on only one example. Therefore, I tested three different query cars from the dataset.

In [ ]:
query_indices = [42, 100, 500]
query_results = []

for idx in query_indices:
    result = find_similar_cars(query_idx=idx, k=10, show_table=False)
    query_results.append({
        'Query Index': idx,
        'Actual Price': result['query_price'],
        'Lower Bound': result['lower_bound'],
        'Upper Bound': result['upper_bound'],
        'Median Price': result['median_price'],
        'IQR': result['iqr'],
        'Interpretation': result['interpretation']
    })

multi_query_df = pd.DataFrame(query_results)
multi_query_df.style.format({
    'Actual Price': '{:,.0f} TL',
    'Lower Bound': '{:,.0f} TL',
    'Upper Bound': '{:,.0f} TL',
    'Median Price': '{:,.0f} TL',
    'IQR': '{:,.0f} TL'
})

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
labels = [f"Query {i}" for i in multi_query_df['Query Index']]
ax.plot(labels, multi_query_df['Actual Price'], marker='o', label='Actual Price')
ax.plot(labels, multi_query_df['Lower Bound'], marker='o', label='Lower Bound')
ax.plot(labels, multi_query_df['Upper Bound'], marker='o', label='Upper Bound')
ax.set_ylabel('Price (TL)')
ax.set_title('Actual Price vs Recommended Price Range for Multiple Query Cars')
ax.legend()
plt.tight_layout()
plt.show()

## 9. Final Interpretation

The kNN model provides a practical and explainable recommendation system. Instead of directly predicting a price with a black-box regression model, it finds vehicles that are similar to the query car and uses their real market prices to calculate a fair selling range.

For the main query car, if the actual price is above the recommended range, the car may be overpriced compared with similar cars. If it is below the range, it may be underpriced. If it is inside the range, the listing price is consistent with the market.

The results may change depending on the selected features and the value of k. A smaller k focuses on very close neighbors but may be sensitive to outliers. A larger k gives a more stable result but may include less similar vehicles. Therefore, comparing k values improves the reliability of the analysis.

## 10. Limitations and Improvement Suggestions

Although the model works successfully, there are some limitations:

- Brand and exact model identity may not be fully represented in the selected features.
- Luxury and economy cars may sometimes appear similar if they share year, mileage, and technical values.
- Outlier prices can affect the recommended range.
- The result depends heavily on correct scaling and feature selection.

Possible improvements:

- Add stronger brand/model-related features if available.
- Compare different distance metrics.
- Remove extreme outlier prices before training.
- Test more query cars and report the average behavior.

## Conclusion

This project successfully applies a kNN-based similarity search to recommend a fair price range for a car. The model is interpretable because every recommendation is based on actual similar cars from the dataset. This makes the result understandable, practical, and useful for real-world vehicle pricing decisions.